# 04a Theme 1 brief: Awareness / Engagement

Short synthesis of `02` (strategy) and `03` (tiering, screening, ablation, SHAP).

For brand / content stakeholders. Observational only (Nike & Adidas TikTok, ~4.4k videos). Not causal.


## 1. Bottom line

Pre-publish features are weak at point-forecasting WER (~3% MAE improvement over brand median). They are useful for ranking candidates with higher top-decile odds (**Lift@10 ≈ 2.39×**). Strategy still comes from within-brand descriptive work in `02`; the model is a screening layer, not a virality predictor.


## 2. Keep the layers separate

| Layer | Where | Answers | Does not answer |
|-------|-------|---------|-----------------|
| Strategy | `02` crosstab / BRI | What looks strong for Nike vs Adidas vs each brand’s median WER | Extreme-tail shortlist |
| Tiering | `03` `qclass_4` | Coarse Low / Mid / High / Top spectrum and library ranking | Precise virality |
| Screening | `03` Top-10% CatBoost + CLIP | Which candidates enrich the top-decile pool (Lift@K) | Brand content playbook |
| Modality test | `03` §9b ablation | Whether SBERT, visual axes, CLIP PCA, or alignment buy Lift | Label direction |
| Interpretation | `03` §10 SHAP | What the screener uses, and signed direction by brand | Causality |


## 3. Strategy findings (`02`): Nike vs Adidas

- Lifestyle / OOTD is the largest and relatively strong overall direction (`vibe_ootd` + lifestyle style).
- Mix differs by brand: Adidas leans lifestyle / GRWM / OOTD; Nike leans technical / performance / promo / sports-action.
- Adidas median WER is higher than Nike in this sample (Mann-Whitney significant).
- Platform-native and collaborative formats tend to beat hard promo (trend, GRWM/OOTD, collaboration above `product_promo` / `official_campaign`).
- Within-brand winners differ: Adidas favors Lifestyle and Vibe/OOTD; Nike favors Technical and Tutorial/Utility (vs each brand’s own baseline).
- Collaboration sits above baseline for both brands (shared opportunity).

Implication: optimize for each brand’s outperforming territories, not only what they already post most.


## 4. Predictive layer (`03`): locked numbers

### 4.1 Why not exact WER regression

| | WER MAE | Notes |
|-|---------|-------|
| Brand-median baseline | ≈ 0.00493 | |
| CatBoost regression | ≈ 0.00479 | ~3% improvement; not useful as a forecaster |

Better use of the same signals: prioritize unusual high performers.

### 4.2 Tiering: `qclass_4`

- Four train-fold quantile classes; score = expected class $\sum i\,P(Y=i)$.
- Acc ≈ 0.33 (vs ~0.25 random); Q4 OvR AUC ≈ 0.63-0.64; Spearman ≈ 0.23.
- Role: library segmentation, not the headline KPI.

### 4.3 Screening: Top-10% (primary) / Top-25% (robustness)

Author `GroupKFold`; labels from train-fold Q90 / Q75 only. CatBoost tuned on Top-25, then reused for Top-10.

| Model | Lift@5 | Lift@10 | Lift@20 | AUC | Prec@10 |
|-------|--------|---------|---------|-----|----------|
| Top-10% (primary) | 3.10× | 2.39× | 2.06× | 0.686 | 0.241 |
| Top-25% (robustness) | 2.41× | 2.33× | 1.85× | 0.676 | 0.566 |

Among model-ranked top 10%, true top-decile rate is about 24% vs ~10% baseline (2.39× enrichment).

### 4.4 Modality ablation (clean ladder, default CatBoost)

| Step | Add | Lift@10 |
|------|-----|--------|
| M0 | Metadata | 1.97 |
| M1 | + engineered text | 2.32 |
| M2 | + SBERT PCA | 2.12 |
| M3a | + visual format/setting axes | 2.03 |
| M3b | + CLIP visual PCA | 2.59 |
| M4 | + alignment | 2.35 |

Engineered text and CLIP PCA earn Lift. Format/setting axes and alignment help less on Lift (still useful for interpretation and for lining up with `02`).


## 5. Screening drivers (SHAP), not the strategy list

What the Top-10% model leans on (importance):
1. CLIP visual semantics + text embeddings (largest pooled share; latent)
2. Creator audience / identity (`creator_tier`, followers)
3. Caption characteristics
4. Video duration and post timing (separate)
5. Visual format/setting; visual-text alignment (in the model; higher alignment associates with lower P(top) in-sample)

For direction vs `02`, use §10c by-brand signed SHAP. Pooled SHAP can blur opposite brand patterns (e.g. Nike tutorial up, Adidas tutorial down).

Do not swap crosstab winners for SHAP importance ranks.


## 6. Operating loop

1. Start from `02` strategy territories.
2. Score candidate creatives with pre-publish features.
3. Use `qclass_4` for coarse tiering and Top-10% score for shortlist priority.
4. Human / brand review of the shortlist.
5. Controlled tests later if you need causal claims.

### Boundaries

- n ≈ 4.4k; observational hashtag/user sample.
- Unseen-author CV is harder than forecasting known creators.
- CLIP coverage ~84%; some taxonomy axes are sparse (~12-23%).
- No claim of virality prediction, and no causal lift from adopting a feature.


## 7. Pointers

| Need | Open |
|------|------|
| Strategy tables & charts | `02_descriptive_crosstabs.ipynb` |
| Full modeling path | `03_engagement_modeling.ipynb` |
| Locked protocol / numbers | `03` §11 + `notebooks/README.md` |
